<h1 align="center">Organization Info</h1>

**Дополнительный материал для выполнения дз**:
- Лукашин Ю.П. Адаптивные методы краткосрочного прогнозирования временных рядов. Финансы и статистика. 2003, главы 1,4,5,7.
- https://otexts.com/fpp2/expsmooth.html


<h1 align="center">Check Questions (5%)</h1>

Ответе на вопросы своими словами (загугленный материал надо пересказать), ответ обоснуйте (напишите и ОБЪЯСНИТЕ формулки если потребуется), если не выходит, то вернитесь к лекции дополнительным материалам:

**Вопрос 1**: Опишите, как изменяется адаптивная способность алгоритма Simple Exponential Smoothing при изменении параметра $\alpha$ от 0 до 1.

<Ответ>
Если $\alpha$ близко к нулю, то у более дальних наблюдений всё ещё сохраняется ощутимый вес ( $\alpha (1-\alpha)^n$ - при альфа у нуля такой вес убывает медленно с ростом $n$) 

Если $\alpha \rightarrow 1$, то мы опираемся на более ранние наблюдения, так как $\alpha (1-\alpha)^n$ быстро убывает с ростом $n$. 

Если $\alpha = 1$, то мы просто говорим, что завтра погода в точности, как вчера (наивно)

**Вопрос 2**: Докажите равенство выражений в $\color{green}{рекуррентной~форме}$ и в $\color{red} {форме~корректировки~на~ошибку}$ для модели Тейла-Вейджа.

$$ \hat y_{t+d} = \hat l_t + \hat b_t\cdot d + \hat s_{t+(d\mod p)-p}. $$

$$ \hat l_t =  \color{green}{\alpha (y_t - \hat s_{t-p}) + (1-\alpha) (\hat l_{t-1} + \hat b_{t-1} )}=\color{red}{\hat l_{t-1} + \hat b_{t-1} + \alpha e_t};$$

$$\hat b_t =  \color{green}{\beta (\hat l_{t} - \hat l_{t-1} ) + (1-\beta) \hat b_{t-1} } = \color{red}{\hat b_{t-1} + \alpha\beta e_t};$$

$$ \hat s_t = \color{green}{\gamma (y_t-\hat l_{t}) + (1-\gamma) \hat s_{t-p} }= \color{red}{\hat s_{t-p} + \gamma(1-\alpha)e_t}.$$

где
$e_t = y_t-\hat y_t$


<Ответ>
$$ \hat l_t =  \alpha (y_t - \hat s_{t-p}) + (1-\alpha) (\hat l_{t-1} + \hat b_{t-1} ) =\hat l_{t-1} + \hat b_{t-1} + \alpha(y_t - \hat s_{t-p} - \hat l_{t-1} - \hat b_{t-1}) =  \hat l_{t-1} + \hat b_{t-1} + \alpha(y_t - \hat {y_t}) \text{ (По определению $\hat {y_t}$)}$$
$$ \hat b_t = \beta (\hat l_{t} - \hat l_{t-1} ) + (1-\beta) \hat b_{t-1} =  \hat b_{t-1} + \beta (\hat l_{t} - \hat l_{t-1} - \hat b_{t-1}) =\hat b_{t-1} + \beta \alpha e_t \text{ (Использовали формулу выше)}$$

$$ \hat s_t = \gamma (y_t-\hat l_{t}) + (1-\gamma) \hat s_{t-p} = \hat s_{t-p} + \gamma (y_t-\hat l_{t} - \hat s_{t-p}) = \hat s_{t-p} + \gamma (y_t- \hat l_{t-1} - \hat b_{t-1} - \alpha e_t - \hat s_{t-p})  = \hat s_{t-p} + \gamma( e_t - \alpha e_t)$$

**Вопрос 3**: Каким следует выбрать параметр сглаживания тренда $\beta$ в модели Хольта (линейный тренд) в случае, когда вы предсказываете временной ряд 1) с плавно меняющимя трендом; 2) стохастически меняющися трендом?

<Ответ> По формуле для $\hat b_t = \beta (\hat l_{t} - \hat l_{t-1} ) + (1-\beta) \hat b_{t-1}$ видно, что при малых $\beta$ $\hat b_t$ близок к $\hat b_{t-1}$, значит для плавно меняющегося тренда малые $\beta$ подойдут, так как предсказания будут более гладкими

А для стохастического лучше взять большие $\beta$, таким образом $\hat l_{t} - \hat l_{t-1}$ влияет больше, модель изменяется быстрее


<h1 align="center"> Practice</h1>

#1. reading data (5%)


In [27]:
# start with this code
import pandas as pd
import numpy as np
from utils import InitExponentialSmoothing, build_forecast, plot_ts_forecast
from utils import qualityMAPE

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
pd.options.plotting.backend = "plotly"
 	
df = pd.read_csv('https://raw.githubusercontent.com/aromanenko/ATSF/main/data/energy_consumption.csv', parse_dates=['Date'])
ts = df[df.id == 2].drop(columns='id').set_index('Date')['2011-01-01':'2014-01-01']

# # Put your code below
ts.plot().update_layout(height=350, width=1350).show()

In [28]:
ts.index

DatetimeIndex(['2011-01-01', '2011-01-02', '2011-01-03', '2011-01-04',
               '2011-01-05', '2011-01-06', '2011-01-07', '2011-01-08',
               '2011-01-09', '2011-01-10',
               ...
               '2013-12-23', '2013-12-24', '2013-12-25', '2013-12-26',
               '2013-12-27', '2013-12-28', '2013-12-29', '2013-12-30',
               '2013-12-31', '2014-01-01'],
              dtype='datetime64[ns]', name='Date', length=1097, freq=None)

# 2. Build the Forecast with  SES (20%)

You need to apply SES model for the ts.
You can use code from seminars or you can write down your own code using any python lib.

Forecast delay $h=1$ for all point in this task.

* 0) Forecast the ts with SES $\alpha=.1$.
* 1) Split the ts to 4 equal parts: find the best param $\alpha$ of SES for the based on 3-rd part of ts (e.g. if time series includes date from 01Jan2020 till 31Dec2020 then 3rd part will be from 1Jul2020 till 30Sep2020). Is the optimal value close to 0 or 1? (use MAPE as a loss function).
* 2) Draw the forecast that correspond to SES with optimial value $\alpha$
     Conclude whether SES can be used for this TS? If can not than explain why.
* 3) Calculate MAPE loss of the best forecast of the ts based on 4-th part of the ts.

---



## 1) Search for the optimal $\alpha$

In [29]:
ALPHA = np.arange(0.1, 1, 0.05)
ESparams = [{'alpha':alpha} for alpha in ALPHA]
FRC_ts = build_forecast(h=1, ts=ts[['EnergyConsumption']], alg_name = "SimpleExponentialSmoothing", alg_title = "ses",params = ESparams)

In [30]:
n = len(ts)//4

In [31]:
# compare ES parameters
qlt_ses = pd.DataFrame(index = ts.columns, columns = FRC_ts.keys())

ix = ts.iloc[2*n:3*n].index
for param_cntr in sorted(qlt_ses.columns):
    frc_ts = FRC_ts[param_cntr]
    qlt_ses[param_cntr],_ = qualityMAPE(ts[['EnergyConsumption']].loc[ix], frc_ts.loc[ix])

qlt_ses[qlt_ses.columns].mean().sort_values()

ses {'alpha': np.float64(0.9500000000000003)}     0.016022
ses {'alpha': np.float64(0.9000000000000002)}     0.016289
ses {'alpha': np.float64(0.8500000000000002)}     0.016561
ses {'alpha': np.float64(0.8000000000000002)}     0.016855
ses {'alpha': np.float64(0.7500000000000002)}     0.017135
ses {'alpha': np.float64(0.7000000000000002)}     0.017404
ses {'alpha': np.float64(0.6500000000000001)}     0.017665
ses {'alpha': np.float64(0.6000000000000002)}     0.017903
ses {'alpha': np.float64(0.5500000000000002)}     0.018104
ses {'alpha': np.float64(0.5000000000000001)}     0.018312
ses {'alpha': np.float64(0.45000000000000007)}    0.018516
ses {'alpha': np.float64(0.40000000000000013)}    0.018692
ses {'alpha': np.float64(0.3500000000000001)}     0.018861
ses {'alpha': np.float64(0.30000000000000004)}    0.019038
ses {'alpha': np.float64(0.25000000000000006)}    0.019277
ses {'alpha': np.float64(0.20000000000000004)}    0.019769
ses {'alpha': np.float64(0.15000000000000002)}    0.0207

In [32]:
qlt_ses[qlt_ses.columns].mean().sort_values().idxmin(), qlt_ses[qlt_ses.columns].mean().min()

("ses {'alpha': np.float64(0.9500000000000003)}",
 np.float64(0.016021604811840135))

0.95 Альфа нам подходит. Значение близко к единице, значит ряд довольно сильно скачет

## 2) draw the forecast with optimial value $\alpha $

In [33]:
FRC_ts = build_forecast(h=1, ts=ts[['EnergyConsumption']], alg_name = "SimpleExponentialSmoothing", alg_title = "ses",params = [{'alpha':0.95}])

In [34]:
plot_ts_forecast(ts[['EnergyConsumption']].iloc[2*n:3*n], FRC_ts["ses {\'alpha\': 0.95}"].iloc[2*n:3*n]
               , ts_num=0, alg_title="ses")

In [35]:
plot_ts_forecast(ts[['EnergyConsumption']], FRC_ts["ses {\'alpha\': 0.95}"]
               , ts_num=0, alg_title="ses")

** Question**
    * Does SES follow to the TS components?

По графику видно, что предсказания получились смещенными

SES при большом альфа просто берет $y$ предыдущего дня, поэтому такой результат. По сути мы ничего хорошего не выучили, значит SES нам не подойдёт

## 3) Calculate loss of the forecast of TS in 4th part of the time series

In [36]:
FRC_ts = build_forecast(h=1, ts=ts[['EnergyConsumption']], alg_name = "SimpleExponentialSmoothing", alg_title = "ses",params = [{'alpha':0.95}])
ix = ts.iloc[3*n: ].index
mape4, _ = qualityMAPE(ts[['EnergyConsumption']].loc[ix], FRC_ts["ses {\'alpha\': 0.95}"].loc[ix])
mape4

EnergyConsumption    0.015192
dtype: float64

Оценка почти не изменилась по сравнению с 3 частью от ряда

# 3. Winters model for Additive Seasonality (25%)
You need to realize ES model for TS with additive seasonality and then apply it to the ts.

You can use code from seminars or you can write down your own code using any python lib.


Forecast delay $h=1$ for all point in this task.

* 1) Realize Additive Winters model
* 2) Split the ts to 4 equal parts: find the best params $\alpha$ (smoothing of level) and $\gamma$ (smoothing of seasonality) for the based on 3-rd part of ts (e.g. if time series includes date from 01Jan2020 till 31Dec2020 then 3rd part will be from 1Jul2020 till 30Sep2020). Use MAPE as a loss function.
* 3) Draw the forecast that correspond optimal values $\alpha$ and $\gamma$ for the whole TS
* 4) Calculate MAPE loss of the best forecast of the ts based on 4-th part of the ts. Compare it with accuracy of SES: is it better?
* 5) Based on results of 3) and 4) conclude whether Additive Winter's ES is appropriate for this TS.

<b>Использовал utils, но пришлось модифицировать код для Adaptive, чтобы убрать варнинги</b>

In [37]:
import itertools


ix = ts.iloc[2*n:3*n].index
season_length = 7
ALPHA = np.arange(0.1, 1, 0.1)
GAMMA = np.arange(0.1, 1, 0.1)
all_pairs = list(itertools.product(ALPHA, GAMMA))
ESparams = [{'alpha':alpha, 'gamma':gamma, 'seasonality_period':season_length} for alpha, gamma in all_pairs]
FRC_ts = build_forecast(h=1, ts=ts[['EnergyConsumption']], alg_name = "AdditiveWintersExponentialSmoothing", alg_title = "AWE",params = ESparams)

In [38]:
qlt_ses = pd.DataFrame(index = ts.columns, columns = FRC_ts.keys())

for param_cntr in sorted(qlt_ses.columns):
    frc_ts = FRC_ts[param_cntr]
    qlt_ses[param_cntr],_ = qualityMAPE(ts[['EnergyConsumption']].loc[ix], frc_ts.loc[ix])

qlt_ses[qlt_ses.columns].mean().sort_values().idxmin(), qlt_ses[qlt_ses.columns].mean().min()

("AWE {'alpha': np.float64(0.9), 'gamma': np.float64(0.1), 'seasonality_period': 7}",
 np.float64(0.009370238689251815))

In [39]:
FRC_ts = build_forecast(h=1, ts=ts[['EnergyConsumption']], alg_name = "AdditiveWintersExponentialSmoothing", alg_title = "AWE",params = [{'alpha':0.9, 'gamma':0.1, 'seasonality_period':season_length}])
plot_ts_forecast(ts[['EnergyConsumption']], FRC_ts["AWE {'alpha': 0.9, 'gamma': 0.1, 'seasonality_period': 7}"]
               , ts_num=0, alg_title="AWE")

In [40]:
FRC_ts = build_forecast(h=1, ts=ts[['EnergyConsumption']].loc[ts.index], alg_name = "AdditiveWintersExponentialSmoothing", alg_title = "AWE",params = [{'alpha':0.9, 'gamma':0.1, 'seasonality_period':7}])
plot_ts_forecast(ts[['EnergyConsumption']].loc[ts.index], FRC_ts["AWE {'alpha': 0.9, 'gamma': 0.1, 'seasonality_period': 7}"]
               , ts_num=0, alg_title="AWE")

In [41]:
FRC_ts = build_forecast(h=1, ts=ts[['EnergyConsumption']], alg_name = "AdditiveWintersExponentialSmoothing", alg_title = "AWE",params = [{'alpha':0.9, 'gamma':0.1, 'seasonality_period':7}])

In [42]:
ix = ts.iloc[3*n: ].index
mape4, _ = qualityMAPE(ts[['EnergyConsumption']].loc[ix], FRC_ts["AWE {'alpha': 0.9, 'gamma': 0.1, 'seasonality_period': 7}"].loc[ix])
print(f'{mape4.iloc[0]:.8f}')

0.00778174


Я взял недельную сезонность для датасета и получил результаты лучше, чем для SES

По полученной ошибке и графикам могу сказать, что AWE подходит для нашей ts - она хорошо выучила недельную сезонность и смогла адаптироваться под изменения, происходящие за год

# 4. Theil-Wage model for TS with linear trend and seasonality (25%)
You need to realize Theil-Wage model and then use it for forecasting the ts.

You can use code from seminars or you can write down your own code using any python lib.


Forecast delay $h=1$ for all point in this task.

* 1) Realize Theil-Wage model
* 2) Split the ts to 4 equal parts: find the best params $\alpha$ (smoothing of level), $\beta$ (smoothing of trend) and $\gamma$ (smoothing of seasonality) for the based on 3-rd part of ts (e.g. if time series includes date from 01Jan2020 till 31Dec2020 then 3rd part will be from 1Jul2020 till 30Sep2020). Use MAPE as a loss function.
* 3) Draw forecast with optimal values $\alpha$, $\beta$ and $\gamma$
* 4) Calculate MAPE loss of the best forecast of the ts based on 4-th part of the ts. Compare it with accuracy of Additive Winters model: is it better than the last one?
* *5) Suggest how can the Theil-Wage model be improved to make accuracy of forecast better?

In [43]:
import itertools


ix = ts.iloc[2*n:3*n].index
season_length = 7
ALPHA = np.arange(0.1, 1, 0.1)
BETA = np.arange(0.1, 1, 0.1)
GAMMA = np.arange(0.1, 1, 0.1)
all_pairs = list(itertools.product(ALPHA, BETA, GAMMA))
ESparams = [{'alpha':alpha, 'beta': beta ,'gamma':gamma, 'seasonality_period':season_length} for alpha, beta, gamma in all_pairs]
FRC_ts = build_forecast(h=1, ts=ts[['EnergyConsumption']], alg_name = "TheilWageExponentialSmoothing", alg_title = "TWS",params = ESparams)

In [44]:
qlt_ses = pd.DataFrame(index = ts.columns, columns = FRC_ts.keys())

for param_cntr in sorted(qlt_ses.columns):
    frc_ts = FRC_ts[param_cntr]
    qlt_ses[param_cntr],_ = qualityMAPE(ts[['EnergyConsumption']].loc[ix], frc_ts.loc[ix])

qlt_ses[qlt_ses.columns].mean().sort_values().idxmin(), qlt_ses[qlt_ses.columns].mean().min()

("TWS {'alpha': np.float64(0.9), 'beta': np.float64(0.1), 'gamma': np.float64(0.1), 'seasonality_period': 7}",
 np.float64(0.009830373917071387))

In [45]:
FRC_ts = build_forecast(h=1, ts=ts[['EnergyConsumption']], alg_name = "TheilWageExponentialSmoothing", alg_title = "TWS",params = [{'alpha':0.9, 'beta':0.1, 'gamma':0.1, 'seasonality_period':season_length}])
plot_ts_forecast(ts[['EnergyConsumption']], FRC_ts["TWS {'alpha': 0.9, 'beta': 0.1, 'gamma': 0.1, 'seasonality_period': 7}"]
               , ts_num=0, alg_title="TWS")

In [46]:
FRC_ts = build_forecast(h=1, ts=ts[['EnergyConsumption']], alg_name = "TheilWageExponentialSmoothing", alg_title = "TWS",params = [{'alpha':0.9, 'beta':0.1, 'gamma':0.1, 'seasonality_period':season_length}])
ix = ts.iloc[3*n: ].index
mape4, _ = qualityMAPE(ts[['EnergyConsumption']].loc[ix], FRC_ts["TWS {'alpha': 0.9, 'beta': 0.1, 'gamma': 0.1, 'seasonality_period': 7}"].loc[ix])
print(f'{mape4.iloc[0]:.8f}')

0.00788543


Результаты очень схожи с прошлой моделью, расхождения в 4 знаке - это может указывать на отсутствие выраженного тренда. Получаем, что для нашей TS усложнять модель нет смысла

Возможно, следует выбрать другую сезонность, чтобы можно было легче уловить тренд

In [83]:
ix = ts.iloc[2*n:3*n].index
season_length = 30
ALPHA = np.arange(0.1, 1, 0.1)
BETA = np.arange(0.1, 1, 0.1)
GAMMA = np.arange(0.1, 1, 0.1)
all_pairs = list(itertools.product(ALPHA, BETA, GAMMA))
ESparams = [{'alpha':alpha, 'beta': beta ,'gamma':gamma, 'seasonality_period':season_length} for alpha, beta, gamma in all_pairs]
FRC_ts = build_forecast(h=1, ts=ts[['EnergyConsumption']], alg_name = "TheilWageExponentialSmoothing", alg_title = "TWS",params = ESparams)
qlt_ses = pd.DataFrame(index = ts.columns, columns = FRC_ts.keys())

for param_cntr in sorted(qlt_ses.columns):
    frc_ts = FRC_ts[param_cntr]
    qlt_ses[param_cntr],_ = qualityMAPE(ts[['EnergyConsumption']].loc[ix], frc_ts.loc[ix])

qlt_ses[qlt_ses.columns].mean().sort_values().idxmin(), qlt_ses[qlt_ses.columns].mean().min()

("TWS {'alpha': np.float64(0.9), 'beta': np.float64(0.1), 'gamma': np.float64(0.2), 'seasonality_period': 30}",
 np.float64(0.019868910228218205))

С таким сезоном вышло ещё хуже

<b>Улучшения:</b>
1) Можно попробовать инициализировать начальные значения для массивов b, s - по графику видно, что начало графиков смазано
2) В обоих методах выше для всего ряда коэффициенты $\alpha, \beta, \gamma$ фиксированны. Возможно можно их обновлять спустя какой то период

# 5. Non-additive model of ES (25%)
You need to realize some ES-model that include non-addive component (or multiplicative trend or multiplicative component) or/and damped-trend component and then use it for forecasting of the ts

You can use code from seminars or you can write down your own code using any python lib.

Forecast delay $h=1$ for all point in this task.

* 1) Realize one of following ES models: ESM(A,M) (t.e. Holt-Winters model), ESM(Ad,M), ESM(M,A), ESM(M,M), ESM(Md,M) model.
* 2) Split the ts to 4 equal parts: find the best params $\alpha$ (smoothing of level)/$\beta$ (smoothing of trend)/$\gamma$ (smoothing of seasonality) for the based on 3-rd part of ts (e.g. if time series includes date from 01Jan2020 till 31Dec2020 then 3rd part will be from 1Jul2020 till 30Sep2020). Use MAPE as a loss function.
Note: if you seelct damped trend model then you can set  $\phi$ value expertly (say $0.98$). (Loss function should be the same as in task 2.)
* 3) Draw forecast with optimal values of it's params.
* 4) Calculate accuracy of the forecast of TS based on 4-th part of the ts. Compare it with accuracy of Additive Winters model and Theil-Wage model, which model is the best?
* 5) Will be results the same if forecas horizon is different (h = seasonlaity period of data)? Please give reasons for your answer.

Я реализую 	Multiplicative Holt-Winters’ method - (A,M)

In [ ]:
def MultiplicativeWintersExponentialSmoothing(x, h, Params):
    T = len(x)
    alpha = Params['alpha']
    beta = Params['beta']
    gamma = Params['gamma']
    p = Params['seasonality_period']
    
    FORECAST = [np.nan]*(T+h)
    l = np.nan 
    b = 0  
    s = []  
    
    for cntr in range(T):
        if not math.isnan(x.iloc[cntr]):
            if math.isnan(l):
                l = x.iloc[cntr]
            if len(s) == 0:
                for i in range(p):
                    s.append(x.iloc[i]/l)
            
            if cntr < p:
                l_new = alpha*(x.iloc[cntr]/ s[cntr]) + (1-alpha)*(l + b)
            else:
                l_new = alpha*(x.iloc[cntr]/ s[cntr - p]) + (1 - alpha)* (l + b)
                s.append(gamma*(x.iloc[cntr]/ l_new) + (1 - gamma)*s[cntr - p])
            
            b = beta*(l_new - l) + (1 - beta)*b
            l = l_new
            
        FORECAST[cntr+h] = (l + h* b) * s[cntr + h - (1 +h // p) * p]
    
    return FORECAST

Добавлю эту функцию в utils, чтобы запускать build_forecast

In [23]:
import itertools

n = len(ts)//4
ix = ts.iloc[2*n:3*n].index
season_length = 7
ALPHA = np.arange(0.1, 1, 0.1)
BETA = np.arange(0.1, 1, 0.1)
GAMMA = np.arange(0.1, 1, 0.1)
all_pairs = list(itertools.product(ALPHA, BETA, GAMMA))
ESparams = [{'alpha':alpha, 'beta': beta ,'gamma':gamma, 'seasonality_period':season_length} for alpha, beta, gamma in all_pairs]
FRC_ts = build_forecast(h=1, ts=ts[['EnergyConsumption']], alg_name = "MultiplicativeWintersExponentialSmoothing", alg_title = "MWES",params = ESparams)

In [24]:
qlt_ses = pd.DataFrame(index = ts.columns, columns = FRC_ts.keys())

for param_cntr in sorted(qlt_ses.columns):
    frc_ts = FRC_ts[param_cntr]
    qlt_ses[param_cntr],_ = qualityMAPE(ts[['EnergyConsumption']].loc[ix], frc_ts.loc[ix])

qlt_ses[qlt_ses.columns].mean().sort_values().idxmin(), qlt_ses[qlt_ses.columns].mean().min()

("MWES {'alpha': np.float64(0.9), 'beta': np.float64(0.1), 'gamma': np.float64(0.30000000000000004), 'seasonality_period': 7}",
 np.float64(0.008779185776016857))

In [25]:
FRC_ts = build_forecast(h=1, ts=ts[['EnergyConsumption']], alg_name = "MultiplicativeWintersExponentialSmoothing", alg_title = "MWES",params = [{'alpha':0.9, 'beta':0.1, 'gamma':0.3, 'seasonality_period':season_length}])
plot_ts_forecast(ts[['EnergyConsumption']], FRC_ts["MWES {'alpha': 0.9, 'beta': 0.1, 'gamma': 0.3, 'seasonality_period': 7}"]
               , ts_num=0, alg_title="MWES")

In [26]:
FRC_ts = build_forecast(h=1, ts=ts[['EnergyConsumption']], alg_name = "MultiplicativeWintersExponentialSmoothing", alg_title = "MWES",params = [{'alpha':0.9, 'beta':0.1, 'gamma':0.3, 'seasonality_period':season_length}])
ix = ts.iloc[3*n: ].index
mape4, _ = qualityMAPE(ts[['EnergyConsumption']].loc[ix], FRC_ts["MWES {'alpha': 0.9, 'beta': 0.1, 'gamma': 0.3, 'seasonality_period': 7}"].loc[ix])
print(f'{mape4.iloc[0]:.8f}')

0.00731805


Получили, что такой метод улучшил предыдущие 2 (0.0073 меньше чем 0.0078 и 0.0077)

<b>ВЫВОД:</b> MWES победил для такого временного ряда с $h=1$

Результаты при замене $h$ будут отличаться, потому что:
1) Чем дальше прогноз, тем больше накапливается ошибка
2) Адаптивный и мультипликативный методы будут эту ошибку учитывать по разному
3) Нужна будет новая сезонность p, если h > 7
4) Для предсказания на завтра тренд особо не требовался, а даже мешал в некоторых методах. Нет гарантий, что такое будет сохраняться для h = 7 или более
5) Графики выглядят как подряд нарисованные параболы - не думаю, что аддитивность поможет хорошо закрыть кривизну, в отличие от умножения